# Initial Setup

In [25]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog verilator gtkwave covered

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:9 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,171 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,733 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 4,030 kB in 3s (1,275 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' 

In [26]:
#@title Setting up PPA estimation

%%bash
set -e
cd /content

URL=$(python3 - <<'PY'
import json, urllib.request
data = json.load(urllib.request.urlopen(
    "https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest"
))
asset = next(
    a for a in data["assets"]
    if "linux-x64" in a["name"] and a["name"].endswith(".tgz")
)
print(asset["browser_download_url"])
PY
)

wget -q -O oss-cad-suite.tgz "$URL"
rm -rf /content/oss-cad-suite /content/oss-cad-suite-install
mkdir -p /content/oss-cad-suite-install
tar -xzf oss-cad-suite.tgz -C /content/oss-cad-suite-install
mv /content/oss-cad-suite-install/oss-cad-suite /content/oss-cad-suite

/content/oss-cad-suite/bin/yosys -V



Yosys 0.70+221 (git sha1 33177a407, Release, Clang /usr/bin/clang++ 21.1.8)


In [27]:
#@title Select Model
# Using OpenAI gpt-4o-mini for this assignment (consistent with earlier submissions in this course)
model_choice = "gpt-4o-mini"
#model_choice = "gpt-5.2"
#model_choice = "gpt-5.2-codex"
#model_choice = "gpt-4o"
#model_choice = "claude-sonnet-4-6"
#model_choice = "gemini-2.5-flash"

In [28]:
#@title Utility functions

import sys
import os
import time
import re
from abc import ABC, abstractmethod

import openai
import anthropic

try:
    import google.genai.errors
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

################################################################################
### LOGGING
################################################################################

class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################

class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        openai.api_key = os.environ["OPENAI_API_KEY"]
        self.client = openai.OpenAI()
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [
            {
                "role": msg["role"] if msg["role"] in ["system", "user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )

        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20

        messages = [
            {
                "role": msg["role"] if msg["role"] in ["user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=messages
                ).content[0].text
                return output

            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)

        print(f"Failed, exceeded max retries {max_retries}")
        return ""


class Gemini(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        if genai is None:
            raise ImportError("google-genai is not installed or could not be imported.")
        self.gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text

        return output


class HuggingFace(AbstractLLM):
    """
    Local Hugging Face model backend.

    Downloads the selected model from the Hugging Face Hub using snapshot_download,
    then loads it with transformers. The model is cached and reused so it is not
    reloaded every time generate_verilog() is called.
    """

    _loaded_model_id = None
    _tokenizer = None
    _model = None
    _local_model_path = None

    def __init__(
        self,
        model_id=None,
        max_new_tokens=4096,
        temperature=0.2,
        top_p=0.95,
        load_in_4bit=False,
        trust_remote_code=True,
        strip_thinking=True,
    ):
        super().__init__()

        self.model_id = model_id or model_choice

        # If the user accidentally wrote "/org/model", convert it to "org/model".
        # Absolute local paths like "/content/my_model" are still allowed.
        if self.model_id.startswith("/") and not self.model_id.startswith("/content/"):
            self.model_id = self.model_id[1:]

        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.load_in_4bit = load_in_4bit
        self.trust_remote_code = trust_remote_code
        self.strip_thinking = strip_thinking

        self.hf_token = (
            os.environ.get("HF_TOKEN")
            or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
            or None
        )

        self.cache_dir = os.environ.get("HF_HOME", "/content/huggingface")

        self._ensure_model_loaded()

    def _get_torch_dtype(self):
        import torch

        if not torch.cuda.is_available():
            return torch.float32

        major, minor = torch.cuda.get_device_capability(0)

        # A100/H100 and newer generally support bf16 well.
        if major >= 8:
            return torch.bfloat16

        # T4/V100 are safer with fp16.
        return torch.float16

    def _ensure_model_loaded(self):
        if HuggingFace._model is not None and HuggingFace._loaded_model_id == self.model_id:
            return

        import torch
        from huggingface_hub import snapshot_download
        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[HuggingFace] Downloading/caching model: {self.model_id}")

        # If this is a local path, do not call snapshot_download.
        if os.path.isdir(self.model_id):
            local_model_path = self.model_id
        else:
            local_model_path = snapshot_download(
                repo_id=self.model_id,
                cache_dir=self.cache_dir,
                token=self.hf_token,
            )

        print(f"[HuggingFace] Loading tokenizer from: {local_model_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            local_model_path,
            trust_remote_code=self.trust_remote_code,
            token=self.hf_token,
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        torch_dtype = self._get_torch_dtype()

        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto",
            "low_cpu_mem_usage": True,
            "token": self.hf_token,
        }

        if torch.cuda.is_available() and self.load_in_4bit:
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=False,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch_dtype,
                bnb_4bit_use_double_quant=True,
            )

            model_kwargs["quantization_config"] = quantization_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        print("[HuggingFace] Loading model. This can take a few minutes the first time.")

        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            **model_kwargs
        )

        model.eval()

        HuggingFace._loaded_model_id = self.model_id
        HuggingFace._tokenizer = tokenizer
        HuggingFace._model = model
        HuggingFace._local_model_path = local_model_path

        print("[HuggingFace] Model loaded successfully.")

    def _format_prompt(self, conversation: Conversation):
        tokenizer = HuggingFace._tokenizer

        messages = []
        for msg in conversation.get_messages():
            role = msg.get("role", "user")
            content = msg.get("content", "")

            if role not in ["system", "user", "assistant"]:
                role = "user"

            messages.append({"role": role, "content": content})

        # Prefer the model's own chat template when available.
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            # Generic fallback for non-chat-template models.
            prompt_parts = []
            for msg in messages:
                prompt_parts.append(f"{msg['role'].upper()}:\n{msg['content']}\n")
            prompt_parts.append("ASSISTANT:\n")
            return "\n".join(prompt_parts)

    def _clean_output(self, text):
        text = text.strip()

        # Helpful for reasoning models that emit <think>...</think>.
        if self.strip_thinking:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

        return text

    def generate(self, conversation: Conversation, num_choices=1):
        import torch

        tokenizer = HuggingFace._tokenizer
        model = HuggingFace._model

        prompt = self._format_prompt(conversation)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=False,
        )

        # With device_map="auto", use the first model parameter's device.
        first_device = next(model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}

        do_sample = self.temperature is not None and self.temperature > 0

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=do_sample,
                temperature=self.temperature if do_sample else None,
                top_p=self.top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        generated_ids = outputs[0][prompt_len:]

        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return self._clean_output(text)


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################

def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No Verilog modules found in response")
        exit(3)

    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type=None, model_id=""):
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    selected_model_id = model_id or model_choice

    if model_type == "ChatGPT":
        model = ChatGPT(model_id=selected_model_id)
    elif model_type == "Claude":
        model = Claude(model_id=selected_model_id)
    elif model_type == "Gemini":
        model = Gemini(model_id=selected_model_id)
    elif model_type == "HuggingFace":
        model = HuggingFace(model_id=selected_model_id)
    else:
        raise ValueError(
            f"Invalid model type: {model_type}. "
            "Expected one of: ChatGPT, Claude, Gemini, HuggingFace."
        )

    return model.generate(conv)

In [29]:
#@title Feedback Loop

import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []
    for p in file_list:
        if not p:
            continue

        try:
            with open(p, "r") as f:
                text = f.read()
                buf.append(text)
                if not text.endswith("\n"):
                    buf.append("\n")
        except FileNotFoundError:
            print(f"[Warning] Previous module file not found and will be skipped: {p}")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the end of long logs, where compiler errors usually appear."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    tail = s[-max_chars:]
    return "[...truncated...]\n" + tail


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a way that works for API models and local HF models.

    ChatGPT/Gemini/HuggingFace can use a system role.
    Claude path in the earlier utility cell maps unsupported roles to user,
    but we keep Claude as user here to avoid role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for your Verilog testbenches.

    Accepts:
      - "All tests passed"
      - "passed!"
      - "Test x passed"

    Rejects:
      - "failed"
      - "Some tests failed"
      - non-empty fatal/error-like stderr
    """
    out = (stdout_text or "")
    err = (stderr_text or "")
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "error running",
        "fatal",
        "syntax error",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    outdir="",
    log=None,
    prev_modules=None,
    model_id="",
):
    """
    Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Key behaviors:
      - The LLM is asked to output ONLY the current module.
      - For compilation, this concatenates:
            [prev_modules...] + [current module]
        into a temporary combined file.
      - After an error, repair iterations only resend:
            (a) previous module version
            (b) most recent compiler/simulation feedback
        to keep prompt size controlled.
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]
    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    if outdir is None:
        outdir = ""

    outdir = outdir.strip()

    if outdir != "":
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir == "":
            return filename
        return os.path.join(outdir, filename)

    # Normalize prev_modules into a list of paths.
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read the testbench so the model can target the test expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is incorrect or has a compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete corrected Verilog module, from module declaration through endmodule. "
        "Use Verilog-2001 only unless explicitly told otherwise."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench and design prompt."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it. Only use it to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: only previous module plus most recent error feedback."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module. "
            "Do not output explanations. "
            "Do not output a testbench. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)
        return conv

    conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    module_file = _outpath(module + ".v")
    combined_file = _outpath(module + "__combined.v")
    sim_output_file = _outpath(module)

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        print(f"\n========== Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        # Generate or repair only the current module.
        response = generate_verilog(conv, model_type, model_id=model_id)

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        conv.add_message("assistant", response)

        # Write raw model response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_" + str(iterations) + ".txt")
        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and write just the current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            last_module_text = response
            conv = _new_conv_repair_only(last_module_text, last_error_text)

            if iterations >= max_iterations:
                timeout = True

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)
            continue

        # Build concatenated source for compilation:
        # previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(module_file, "r") as f:
            curr_text = f.read()

        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)
            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # Compile with iVerilog.
        compile_cmd = [
            "iverilog",
            "-o",
            sim_output_file,
            combined_file,
            testbench,
        ]

        proc = subprocess.run(
            compile_cmd,
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            last_error_text = (
                "iverilog failed.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        elif proc.stderr.strip() != "":
            # Treat warnings as repair feedback.
            # This keeps generated modules cleaner for later synthesis/PPA.
            status = "Warnings compiling testbench"
            print(status)

            last_error_text = (
                "iverilog produced warnings.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        else:
            # Run simulation.
            run_cmd = ["vvp", sim_output_file]

            proc = subprocess.run(
                run_cmd,
                capture_output=True,
                text=True,
            )

            result_stdout = proc.stdout or ""
            result_stderr = proc.stderr or ""

            print(result_stdout)

            if proc.returncode != 0:
                status = "Error running testbench"
                print(status)

                last_error_text = (
                    "vvp simulation failed.\n\n"
                    "Command:\n"
                    + " ".join(run_cmd)
                    + "\n\nstdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            elif not _simulation_passed(result_stdout, result_stderr):
                status = "Testbench did not report success"
                print(status)

                last_error_text = (
                    "Simulation completed, but the output did not indicate success.\n\n"
                    "stdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            else:
                status = "Testbench ran successfully"
                print(status)

                last_error_text = ""
                success = True

        # Write bounded iteration log.
        with open(_outpath("log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

            if last_error_text:
                file.write("\nMost recent feedback:\n")
                file.write(_truncate_text(last_error_text))

        if not success:
            # Keep future repair prompts small.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("\n========== Feedback Loop Complete ==========")
    print("Final status: ", status)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return (
        np.sum(timelist_total),
        np.sum(timelist_gen),
        np.sum(timelist_error),
    )

In [30]:
#@title Hierarchical Loop

import os
import re
import numpy as np


def _normalize_to_list(x):
    """Normalize None/string/list/tuple into a list."""
    if x is None or x == "":
        return []
    if isinstance(x, str):
        return [x]
    return list(x)


def _resolve_prev_module_path(item):
    """
    Resolve previous module reference into a .v path.

    Supported forms:
      - "mux2_1"              -> "./mux2_1/mux2_1.v"
      - "./mux2_1/mux2_1.v"   -> "./mux2_1/mux2_1.v"
      - ("mux2_1", "...", ...) -> "./mux2_1/mux2_1.v"
    """
    if isinstance(item, (list, tuple)):
        item = item[0]

    item = str(item)

    if item.endswith(".v"):
        return item

    return os.path.join("./" + item, item + ".v")


def _extract_verilog_interfaces(verilog_text):
    """
    Extract module declarations/interfaces from Verilog text.

    This is useful for hierarchical prompting because the current module usually
    only needs previous module names and port lists, not full implementations.
    """
    if not verilog_text:
        return ""

    # Match:
    #   module name (...);
    #   module name #(parameters...) (...);
    pattern = r'\bmodule\s+[A-Za-z_][A-Za-z0-9_]*\s*(?:#\s*\(.*?\)\s*)?\(.*?\)\s*;'

    matches = re.findall(pattern, verilog_text, flags=re.DOTALL)

    if not matches:
        return ""

    cleaned = []
    for m in matches:
        cleaned.append(m.strip() + "\nendmodule")

    return "\n\n".join(cleaned) + "\n"


def _read_prev_modules_for_prompt(prev_verilog_files, context_mode="interfaces", max_chars=24000):
    """
    Read previous modules for LLM context.

    context_mode:
      - "interfaces": include only module declarations, best for HF/local models.
      - "full": include full previous module bodies.
      - "none": include no previous code in the prompt.

    Compilation still receives the full previous .v files through verilog_loop(prev_modules=...).
    """
    if context_mode == "none":
        return ""

    chunks = []

    for p in prev_verilog_files:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found for prompt context: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        if context_mode == "interfaces":
            txt = _extract_verilog_interfaces(txt)

        if txt.strip():
            chunks.append("// File: " + p + "\n" + txt.strip() + "\n")

    prev_text = "\n".join(chunks)

    if len(prev_text) > max_chars:
        # Keep the most recent context, because later modules tend to depend on recent modules.
        prev_text = "[...previous module context truncated...]\n" + prev_text[-max_chars:]

    return prev_text


def _call_verilog_loop_compatible(
    prompt,
    module,
    testbench,
    max_iterations,
    model,
    outdir,
    log,
    prev_verilog_files,
    model_id="",
):
    """
    Calls the updated verilog_loop with model_id support.
    Falls back to the older signature if needed.
    """
    try:
        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
            model_id=model_id,
        )
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] verilog_loop does not accept model_id. Falling back to old signature.")

        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
        )


def hier_gen(
    submods,
    prevsubmods="",
    max_iterations=10,
    model_type=None,
    model_id="",
    testbench_template="./{module}tb.v",
    prompt_prev_context_chars=24000,
    prev_context_mode="interfaces",
):
    """
    Hierarchically generate a design by building submodules in order.

    submods:
      list of tuples/lists:
        [
          (file_basename, human_name, io_string),
          ...
        ]

      Example:
        [
          ("mux2_1", "2-to-1 mux", "input wire a, input wire b, input wire sel, output wire y"),
          ("mux4_1", "4-to-1 mux", "input wire [3:0] in, input wire [1:0] sel, output wire y"),
        ]

    prevsubmods:
      Optional previous module references already generated before this call.

      Supported:
        - ""
        - None
        - "mux2_1"
        - ["mux2_1", "mux4_1"]
        - ["./mux2_1/mux2_1.v"]
        - previous submod tuples/lists

    model_type:
      Optional. If None, uses os.environ["MODEL"].

      Supported by the earlier utility cell:
        - "ChatGPT"
        - "Claude"
        - "Gemini"
        - "HuggingFace"

    model_id:
      Optional model override passed through to verilog_loop/generate_verilog.

      Example:
        model_id="nyu-dice-lab/VeriThoughts-Reasoning-14B"

    testbench_template:
      Default assumes testbenches are named:
        ./module_nametb.v

      You can also use:
        "./testbenches/{module}_tb.v"

    prev_context_mode:
      - "interfaces": best default, includes previous module port declarations only.
      - "full": includes full previous module code in prompt.
      - "none": does not include previous modules in prompt.

      Regardless of this setting, previous modules are still passed to iVerilog
      through verilog_loop(prev_modules=...).
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_context_modes = ["interfaces", "full", "none"]
    if prev_context_mode not in valid_context_modes:
        raise ValueError(
            f"Invalid prev_context_mode: {prev_context_mode}. "
            f"Expected one of: {valid_context_modes}"
        )

    totaltime = []
    gentime = []
    errortime = []

    prev_verilog_files = []

    # Add previously generated external modules, if any.
    for s in _normalize_to_list(prevsubmods):
        p = _resolve_prev_module_path(s)
        if os.path.exists(p):
            prev_verilog_files.append(p)
        else:
            print(f"[Warning] prevsubmod file does not exist and will be skipped: {p}")

    if not submods:
        print("[Warning] No submodules provided.")
        return (0, 0, 0)

    overall = submods[-1][1]

    for i in range(len(submods)):
        fcurr = submods[i][0]   # file/module basename
        curr = submods[i][1]    # human-readable name
        iocurr = submods[i][2]  # port list string

        outdir = "./" + fcurr
        os.makedirs(outdir, exist_ok=True)

        prev_text = _read_prev_modules_for_prompt(
            prev_verilog_files,
            context_mode=prev_context_mode,
            max_chars=prompt_prev_context_chars,
        )

        if i == 0 and len(prev_verilog_files) == 0:
            prompt = (
                "// We will be generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// IMPORTANT: Output ONLY the module requested below.\n"
                "// Do NOT generate any previous modules, future modules, explanations, or testbenches.\n"
                "// Use Verilog-2001 only.\n"
                "// Begin by generating: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )
        else:
            prompt = (
                "// We are generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// Use Verilog-2001 only.\n\n"
                "// Previously generated modules are shown below for reference only.\n"
                "// You may instantiate them, but you must NOT repeat their definitions.\n\n"
                + prev_text
                + "\n"
                "// IMPORTANT: Output ONLY the new module requested below.\n"
                "// Do NOT repeat any previous modules.\n"
                "// Do NOT generate explanations, Markdown fences, or a testbench.\n"
                "// Now generate: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )

        module = fcurr
        testbench = testbench_template.format(module=fcurr, name=fcurr)
        model = model_type
        log = os.path.join(outdir, "log.txt")

        if not os.path.exists(testbench):
            print(f"[Warning] Testbench file does not exist yet: {testbench}")
            print("[Warning] verilog_loop will still run, but iVerilog will fail unless the file exists.")

        total, gen, error = _call_verilog_loop_compatible(
            prompt=prompt,
            module=module,
            testbench=testbench,
            max_iterations=max_iterations,
            model=model,
            outdir=outdir,
            log=log,
            prev_verilog_files=prev_verilog_files,
            model_id=model_id,
        )

        print("Module ", i + 1, " done: ", curr)

        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)

        # Add the current module file to the compilation context for subsequent modules.
        curr_module_file = os.path.join(outdir, module + ".v")

        if os.path.exists(curr_module_file):
            prev_verilog_files.append(curr_module_file)
        else:
            print(f"[Warning] Expected generated module file not found: {curr_module_file}")

    overall_total = np.sum(totaltime)
    overall_gen = np.sum(gentime)
    overall_error = np.sum(errortime)

    print("Overall Total time: ", overall_total)
    print("Overall Generation Time: ", overall_gen)
    print("Overall Error handling time: ", overall_error)

    return (overall_total, overall_gen, overall_error)

In [31]:
#@title PPA Optimizer Loop

#@title PPA Optimization Loop (Yosys no-liberty + iVerilog correctness)

import subprocess
import sys
import os
import time
import math
import json
import re
import shutil
import numpy as np


################################################################################
### GENERAL HELPERS
################################################################################

def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []

    for p in file_list:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found and will be skipped: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        buf.append(txt)

        if txt and not txt.endswith("\n"):
            buf.append("\n")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the tail of long logs."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    return "[...truncated...]\n" + s[-max_chars:]


def _shell_quote_for_yosys(path):
    """Quote filenames for use inside a .ys script."""
    return '"' + str(path).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=None):
    try:
        return int(x)
    except Exception:
        return default


def _first_regex(text, patterns, cast=float):
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            try:
                return cast(m.group(1))
            except Exception:
                pass

    return None


def _normalize_clk_signal(clk_signal):
    """Treat None, '', 'None', 'none', etc. as no clock."""
    if clk_signal is None:
        return None

    s = str(clk_signal).strip()

    if s == "" or s.lower() == "none":
        return None

    return s


def _has_any_parsed_metric(metrics):
    if not metrics:
        return False

    keys = [
        "transistor_est",
        "area_cmos",
        "num_cells",
        "num_wire_bits",
        "timing_proxy",
    ]

    return any(metrics.get(k) is not None for k in keys)


def _is_verbose(verbose):
    return str(verbose).strip().lower() in ["1", "true", "yes", "y"]


def _write_text(path, text):
    with open(path, "w") as f:
        f.write(text)


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a provider-compatible way.

    ChatGPT, Gemini, and HuggingFace can use the system role through the
    Conversation abstraction from the previous utility cell.

    Claude is kept as user to avoid unsupported-role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _generate_verilog_compatible(conv, model_type, model_id=""):
    """
    Calls generate_verilog with model_id support when available.
    Falls back to the older two-argument signature if needed.
    """
    try:
        return generate_verilog(conv, model_type, model_id=model_id)
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] generate_verilog does not accept model_id. Falling back to old signature.")
        return generate_verilog(conv, model_type)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for Verilog testbenches.

    Accepts:
      - All tests passed
      - passed!
      - Test x passed
      - Test x passed!

    Rejects:
      - failed
      - Some tests failed
      - fatal/error markers
    """
    out = stdout_text or ""
    err = stderr_text or ""
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "fatal",
        "syntax error",
        "error:",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


################################################################################
### YOSYS PARSING
################################################################################

def _parse_yosys_stat_text(stat_text):
    """
    Parse Yosys `stat -tech cmos` output.

    Supports both styles:
      Older style:
        Number of wires: ...
        Number of cells: ...
        Chip area for module ...: ...

      Newer compact style:
        6 wires
        165 wire bits
        61 cells
        Estimated number of transistors: 722
    """
    metrics = {}

    metrics["num_wires"] = _first_regex(
        stat_text,
        [
            r"Number of wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wires\s*$",
        ],
        int,
    )

    metrics["num_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wire bits\s*$",
        ],
        int,
    )

    metrics["num_public_wires"] = _first_regex(
        stat_text,
        [
            r"Number of public wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wires\s*$",
        ],
        int,
    )

    metrics["num_public_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of public wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wire bits\s*$",
        ],
        int,
    )

    metrics["num_memories"] = _first_regex(
        stat_text,
        [
            r"Number of memories:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memories\s*$",
        ],
        int,
    )

    metrics["num_memory_bits"] = _first_regex(
        stat_text,
        [
            r"Number of memory bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memory bits\s*$",
        ],
        int,
    )

    metrics["num_processes"] = _first_regex(
        stat_text,
        [
            r"Number of processes:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+processes\s*$",
        ],
        int,
    )

    metrics["num_cells"] = _first_regex(
        stat_text,
        [
            r"Number of cells:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+cells\s*$",
        ],
        int,
    )

    transistor_est = _first_regex(
        stat_text,
        [
            r"Estimated number of transistors:\s*([0-9]+(?:\.[0-9]+)?)",
            r"Chip area for module .*?:\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["transistor_est"] = transistor_est

    # Backward-compatible area-like proxy.
    metrics["area_cmos"] = transistor_est

    return metrics


def _parse_yosys_timeest_text(time_text):
    """
    Parse a timing proxy from Yosys timeest output.

    Returns None if no timing proxy can be found.
    """
    metrics = {}

    if re.search(r"No domain .* in module", time_text or "", flags=re.IGNORECASE):
        metrics["timing_proxy"] = None
        return metrics

    timing_proxy = _first_regex(
        time_text,
        [
            r"critical path.*?([0-9]+(?:\.[0-9]+)?)\s*AIG",
            r"([0-9]+(?:\.[0-9]+)?)\s*AIG nodes",
            r"([0-9]+(?:\.[0-9]+)?)\s*nodes.*critical path",
            r"Estimated.*?([0-9]+(?:\.[0-9]+)?)",
            r"Delay\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["timing_proxy"] = timing_proxy

    return metrics


def _format_metrics(metrics):
    if metrics is None:
        return "None"

    parts = []

    if metrics.get("transistor_est") is not None:
        parts.append("transistor_est={:.0f}".format(metrics["transistor_est"]))
    elif metrics.get("area_cmos") is not None:
        parts.append("area_cmos={:.3f}".format(metrics["area_cmos"]))

    if metrics.get("timing_proxy") is not None:
        parts.append("timing_proxy={:.3f}".format(metrics["timing_proxy"]))

    if metrics.get("num_cells") is not None:
        parts.append("num_cells={}".format(metrics["num_cells"]))

    if metrics.get("num_wire_bits") is not None:
        parts.append("wire_bits={}".format(metrics["num_wire_bits"]))

    return ", ".join(parts) if parts else "No parsed metrics"


################################################################################
### YOSYS ESTIMATION
################################################################################

def yosys_estimate_no_liberty(verilog_file, top_module, clk_signal=None, outdir=""):
    """
    Run a coarse no-liberty Yosys estimate.

    Uses:
      - stat -top <top> -tech cmos
      - optional timeest -clk <clk>

    Returns:
      {
        "ok": bool,
        "metrics": dict,
        "stat_text": str,
        "time_text": str,
        "yosys_stdout": str,
        "yosys_stderr": str,
        "error": str
      }
    """
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    clk_signal = _normalize_clk_signal(clk_signal)

    script_path = os.path.join(outdir, "ppa_estimate.ys") if outdir else "ppa_estimate.ys"
    stat_path = os.path.join(outdir, "ppa_stat.txt") if outdir else "ppa_stat.txt"
    time_path = os.path.join(outdir, "ppa_time.txt") if outdir else "ppa_time.txt"

    ys = []
    ys.append("read_verilog {}".format(_shell_quote_for_yosys(verilog_file)))
    ys.append("hierarchy -check -top {}".format(top_module))
    ys.append("proc")
    ys.append("opt")
    ys.append("memory")
    ys.append("opt")
    ys.append("fsm")
    ys.append("opt")
    ys.append("techmap")
    ys.append("opt")
    ys.append("clean")
    ys.append(
        "tee -o {} stat -top {} -tech cmos".format(
            _shell_quote_for_yosys(stat_path),
            top_module,
        )
    )

    if clk_signal is not None:
        ys.append("flatten")
        ys.append(
            "tee -o {} timeest -clk {}".format(
                _shell_quote_for_yosys(time_path),
                clk_signal,
            )
        )

    with open(script_path, "w") as f:
        f.write("\n".join(ys) + "\n")

    proc = subprocess.run(
        ["yosys", "-s", script_path],
        capture_output=True,
        text=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    result = {
        "ok": False,
        "metrics": {},
        "stat_text": "",
        "time_text": "",
        "yosys_stdout": stdout,
        "yosys_stderr": stderr,
        "error": "",
    }

    if proc.returncode != 0:
        result["error"] = "Yosys failed.\nstdout:\n{}\nstderr:\n{}".format(stdout, stderr)
        return result

    stat_text = ""
    time_text = ""

    if os.path.exists(stat_path):
        with open(stat_path, "r") as f:
            stat_text = f.read()

    if os.path.exists(time_path):
        with open(time_path, "r") as f:
            time_text = f.read()

    metrics = _parse_yosys_stat_text(stat_text)
    metrics.update(_parse_yosys_timeest_text(time_text))

    if metrics.get("transistor_est") is not None:
        metrics["power_proxy"] = metrics["transistor_est"]
    else:
        metrics["power_proxy"] = metrics.get("num_cells")

    result["ok"] = True
    result["metrics"] = metrics
    result["stat_text"] = stat_text
    result["time_text"] = time_text

    if not _has_any_parsed_metric(metrics):
        result["error"] = "Yosys ran successfully, but no metrics were parsed."

    return result


################################################################################
### PPA SCORING
################################################################################

def _component_ratio(candidate_value, baseline_value):
    if candidate_value is None or baseline_value is None:
        return None

    if baseline_value == 0:
        return None

    return float(candidate_value) / float(baseline_value)


def _relative_ppa_score(
    candidate_metrics,
    baseline_metrics,
    area_weight=0.60,
    timing_weight=0.30,
    power_weight=0.10,
):
    """
    Lower is better.

    Uses relative ratios against the current best correct design.
    """
    used_weight = 0.0
    score = 0.0

    cand_area = candidate_metrics.get("transistor_est", candidate_metrics.get("area_cmos"))
    base_area = baseline_metrics.get("transistor_est", baseline_metrics.get("area_cmos"))

    area_ratio = _component_ratio(cand_area, base_area)

    if area_ratio is not None:
        score += area_weight * area_ratio
        used_weight += area_weight

    timing_ratio = _component_ratio(
        candidate_metrics.get("timing_proxy"),
        baseline_metrics.get("timing_proxy"),
    )

    if timing_ratio is not None:
        score += timing_weight * timing_ratio
        used_weight += timing_weight

    power_ratio = _component_ratio(
        candidate_metrics.get("power_proxy"),
        baseline_metrics.get("power_proxy"),
    )

    if power_ratio is not None:
        score += power_weight * power_ratio
        used_weight += power_weight

    if used_weight == 0.0:
        return math.inf

    return score / used_weight


def _is_better_ppa(candidate_metrics, best_metrics, min_improvement=0.0025):
    """
    Returns True only when candidate has parsed metrics and is better.
    """
    if not _has_any_parsed_metric(candidate_metrics):
        return False

    if best_metrics is None or not _has_any_parsed_metric(best_metrics):
        return True

    cand_score = _relative_ppa_score(candidate_metrics, best_metrics)

    # Best-vs-itself is effectively 1.0, so lower is better.
    return cand_score < (1.0 - min_improvement)


################################################################################
### IVERILOG CHECK
################################################################################

def _compile_and_run_iverilog(module_name, combined_file, testbench, outdir):
    """
    Return:
      {
        "ok": bool,
        "status": str,
        "error_text": str,
        "stdout": str,
        "stderr": str
      }
    """
    exe_path = os.path.join(outdir, module_name) if outdir else module_name

    proc = subprocess.run(
        ["iverilog", "-o", exe_path, combined_file, testbench],
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        return {
            "ok": False,
            "status": "Error compiling testbench",
            "error_text": (
                "iverilog failed.\n"
                "Command:\n"
                + " ".join(["iverilog", "-o", exe_path, combined_file, testbench])
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            ),
            "stdout": proc.stdout or "",
            "stderr": proc.stderr or "",
        }

    warn_text = ""

    if proc.stderr and proc.stderr.strip():
        warn_text = (
            "iverilog produced warnings.\n"
            "stderr:\n"
            + (proc.stderr or "")
            + "\nstdout:\n"
            + (proc.stdout or "")
        )

    sim = subprocess.run(
        ["vvp", exe_path],
        capture_output=True,
        text=True,
    )

    result_stdout = sim.stdout or ""
    result_stderr = sim.stderr or ""

    if sim.returncode != 0:
        return {
            "ok": False,
            "status": "Error running testbench",
            "error_text": (
                "vvp simulation failed.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    if not _simulation_passed(result_stdout, result_stderr):
        return {
            "ok": False,
            "status": "Testbench did not report success",
            "error_text": (
                "Simulation completed, but the output did not indicate success.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    return {
        "ok": True,
        "status": "Testbench ran successfully",
        "error_text": warn_text,
        "stdout": result_stdout,
        "stderr": result_stderr,
    }


################################################################################
### MAIN PPA LOOP
################################################################################

def ppa_verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    clk_signal=None,
    outdir="",
    log=None,
    prev_modules=None,
    verbose="0",
    model_id="",
    keep_best_as_module=True,
):
    """
    Generate functionally correct Verilog, then iteratively improve coarse no-liberty
    PPA using Yosys estimates while preserving correctness.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Assumptions:
      - generate_verilog(conv, model_type, model_id=...) exists from the utility cell
      - Conversation(log_file=...) exists
      - write_code_blocks_to_file(response, module, path) exists
      - yosys, iverilog, and vvp are in PATH

    Strategy:
      1. Generate a candidate.
      2. Check correctness with iVerilog/vvp.
      3. If correct, estimate PPA with Yosys.
      4. Keep the best correct candidate seen so far.
      5. Ask the LLM to improve the best correct candidate further.
      6. If a candidate fails, repair using only the failed module plus latest error.
    """
    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]

    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    outdir = "" if outdir is None else str(outdir).strip()

    if outdir:
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir:
            return os.path.join(outdir, filename)
        return filename

    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    tb_text = ""

    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog-2001 code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete Verilog module from module declaration through endmodule. "
        "Preserve the exact module name and port interface. "
        "Preserve functionality required by the testbench. "
        "Optimize for lower coarse PPA under Yosys without a liberty file: "
        "reduce logic depth, reduce unnecessary cells, share repeated logic, simplify control, "
        "avoid redundant muxing, avoid unnecessary intermediate signals, and keep widths disciplined."
    )

    def _new_conv_initial():
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile and simulate your module. "
                "Do NOT generate a testbench or modify it. Use it only to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n"
            )

        conv.add_message(
            "user",
            design_prompt
            + "\n\n"
            "Also optimize the module for better coarse Yosys-estimated PPA without a liberty file. "
            "Output ONLY the full Verilog-2001 module. "
            "Do not use Markdown fences."
        )

        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog-2001 module so it compiles and passes the testbench. "
            "Preserve the exact interface and intended behavior. "
            "Keep PPA reasonably optimized, but correctness is the priority for this step. "
            "Output ONLY the full corrected module. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation/Yosys output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    def _new_conv_optimize_from_best(best_module_text, best_metrics, last_candidate_metrics=None):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Starting from the following functionally correct Verilog-2001 module, "
            "produce a new candidate that preserves the exact same behavior and interface "
            "but improves coarse Yosys-estimated PPA without a liberty file.\n\n"
            "Current best metrics:\n"
            + _format_metrics(best_metrics)
            + "\n\n"
        )

        if last_candidate_metrics is not None:
            prompt += (
                "Most recent candidate metrics:\n"
                + _format_metrics(last_candidate_metrics)
                + "\n\n"
            )

        prompt += (
            "Optimization priorities:\n"
            "1. Lower stat -tech cmos transistor_est / area proxy\n"
            "2. Lower timing proxy, if available\n"
            "3. Fewer cells and wire bits\n\n"
            "Rules:\n"
            "- Do not add helper modules.\n"
            "- Do not change functionality.\n"
            "- Do not change the module name or port interface.\n"
            "- Use Verilog-2001 only.\n"
            "- Output ONLY the full module.\n"
            "- Do not use Markdown fences.\n\n"
            "Current best module:\n"
            "```verilog\n"
            + best_module_text
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    # Timing
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    # Files
    module_file = _outpath(module + ".v")
    best_module_file = _outpath(module + "_best.v")
    combined_file = _outpath(module + "__combined.v")

    # Best-so-far state
    best_module_text = None
    best_metrics = None
    best_iter = None

    # Latest state
    conv = _new_conv_initial()
    last_error_text = ""
    last_candidate_metrics = None

    iterations = 0
    timeout = False

    while not timeout:
        print(f"\n========== PPA Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        response = _generate_verilog_compatible(
            conv,
            model_type,
            model_id=model_id,
        )

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        try:
            conv.add_message("assistant", response)
        except Exception:
            pass

        # Save raw response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_{:03d}.txt".format(iterations))

        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and persist current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            conv = _new_conv_repair_only(response, last_error_text)

            candidate_metrics = None

            log_path = _outpath("log_iter_{:03d}.txt".format(iterations))
            with open(log_path, "w") as f:
                f.write("Iteration status: {}\n".format(status))
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)

            if iterations >= max_iterations:
                timeout = True

            continue

        with open(module_file, "r") as f:
            curr_text = f.read()

        # Concatenate previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(combined_file, "w") as f:
            f.write(prev_text)

            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")

            f.write(curr_text)

            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # 1. Functional check with iVerilog/vvp.
        iverilog_result = _compile_and_run_iverilog(
            module_name=module,
            combined_file=combined_file,
            testbench=testbench,
            outdir=outdir,
        )

        status = iverilog_result["status"]
        accepted = False
        candidate_metrics = None

        if not iverilog_result["ok"]:
            last_error_text = iverilog_result["error_text"]
            conv = _new_conv_repair_only(curr_text, last_error_text)

        else:
            # 2. Coarse PPA estimate with Yosys.
            ppa_dir = _outpath("iter_{:03d}_ppa".format(iterations))
            os.makedirs(ppa_dir, exist_ok=True)

            ppa_result = yosys_estimate_no_liberty(
                verilog_file=combined_file,
                top_module=module,
                clk_signal=clk_signal,
                outdir=ppa_dir,
            )

            if _is_verbose(verbose):
                print("YOSYS OK:", ppa_result["ok"])
                print("YOSYS STDOUT HEAD:\n", (ppa_result["yosys_stdout"] or "")[:3000])
                print("YOSYS STDERR HEAD:\n", (ppa_result["yosys_stderr"] or "")[:3000])
                print("STAT TEXT HEAD:\n", repr((ppa_result["stat_text"] or "")[:1000]))
                print("TIME TEXT HEAD:\n", repr((ppa_result["time_text"] or "")[:1000]))
                print("PARSED METRICS:", ppa_result["metrics"])

            if not ppa_result["ok"]:
                status = "Yosys estimation failed"
                last_error_text = ppa_result["error"]
                conv = _new_conv_repair_only(curr_text, last_error_text)

            else:
                candidate_metrics = ppa_result["metrics"]
                last_candidate_metrics = candidate_metrics

                if not _has_any_parsed_metric(candidate_metrics):
                    status = "Yosys ran but returned no parsable metrics"

                    if best_module_text is not None and best_metrics is not None:
                        conv = _new_conv_optimize_from_best(
                            best_module_text=best_module_text,
                            best_metrics=best_metrics,
                            last_candidate_metrics=None,
                        )
                    else:
                        conv = _new_conv_initial()

                elif _is_better_ppa(candidate_metrics, best_metrics):
                    best_module_text = curr_text
                    best_metrics = candidate_metrics
                    best_iter = iterations
                    accepted = True

                    _write_text(best_module_file, best_module_text)

                    status = "Accepted new best candidate"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text,
                        best_metrics=best_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

                else:
                    status = "Candidate correct but did not improve best PPA"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text if best_module_text is not None else curr_text,
                        best_metrics=best_metrics if best_metrics is not None else candidate_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

        # Write iteration log.
        log_path = _outpath("log_iter_{:03d}.txt".format(iterations))

        with open(log_path, "w") as f:
            try:
                f.write("\n".join(str(i) for i in conv.get_messages()))
            except Exception:
                f.write("(Conversation logging unavailable)\n")

            f.write("\n\nIteration status: {}\n".format(status))

            if candidate_metrics is not None:
                f.write("Candidate metrics: {}\n".format(_format_metrics(candidate_metrics)))

            if best_metrics is not None:
                f.write("Best metrics so far: {}\n".format(_format_metrics(best_metrics)))
                f.write("Best iteration so far: {}\n".format(best_iter))

            if last_error_text:
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

        print("Iteration {}: {}".format(iterations, status))

        if candidate_metrics is not None:
            print("  Candidate metrics:", _format_metrics(candidate_metrics))

        if best_metrics is not None:
            print("  Best metrics so far:", _format_metrics(best_metrics))
            print("  Best iteration so far:", best_iter)

        iterations += 1

        end_time = time.time()

        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

        if iterations >= max_iterations:
            timeout = True

    # Optional: leave module.v as the best candidate, not the last failed/non-improved candidate.
    if keep_best_as_module and best_module_text is not None:
        _write_text(module_file, best_module_text)
        print(f"Best candidate copied to: {module_file}")

    print("\n========== PPA Optimization Complete ==========")
    print("Best file: ", best_module_file if best_module_text is not None else None)
    print("Best metrics: ", _format_metrics(best_metrics) if best_metrics is not None else None)
    print("Best iteration: ", best_iter)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return {
        "best_file": best_module_file if best_module_text is not None else None,
        "best_metrics": best_metrics,
        "best_iteration": best_iter,
        "iterations": iterations,
        "total_time": float(np.sum(timelist_total)),
        "generation_time": float(np.sum(timelist_gen)),
        "error_handling_time": float(np.sum(timelist_error)),
    }

# Setting the API Key

In [32]:
### Read the API key from Colab Secrets (never hardcoded in the notebook)
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["MODEL"] = "ChatGPT"
print("Model: " + os.environ["MODEL"] + ", model_id: " + model_choice)

Model: ChatGPT, model_id: gpt-4o-mini


#Mux Hierarchy Example

In [33]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/mux/mux2_1tb.v ./mux2_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux4_1tb.v ./mux4_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux8_1tb.v ./mux8_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux16_1tb.v ./mux16_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux32_1tb.v ./mux32_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux64_1tb.v ./mux64_1tb.v

fatal: destination path 'ROME-LLM' already exists and is not an empty directory.


In [34]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

In [35]:
hier_gen(submodules)


========== Iteration 0 / 10 ==========
Test           0 passed
Test           1 passed
Test           2 passed
Test           3 passed
Test 4 passed
Test 5 passed!
./mux2_1tb.v:60: $finish called at 60000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.064279794692993
Generation time:  1.9680802822113037
Error handling time:  0.09619903564453125
Module  1  done:  2-to-1 multiplexer

========== Iteration 0 / 10 ==========
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed!
./mux4_1tb.v:40: $finish called at 90000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.166085720062256
Generation time:  2.0874922275543213
Error handling time:  0.07859277725219727
Module  2  done:  4-to-1 multiplexer

========== Iteration 0 / 10 ==========
Test 0 passed
Test 3 passed
Test 6 passed
Test 7 passed!
./mux8_1tb.v:56: $fin

(np.float64(26.776142597198486),
 np.float64(26.21324396133423),
 np.float64(0.562894344329834))

In [36]:
#@title Run PPA optimizer on hierarchical mux design
import os
import json

submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "mux2_1":  "./mux2_1tb.v",
    "mux4_1":  "./mux4_1tb.v",
    "mux8_1":  "./mux8_1tb.v",
    "mux16_1": "./mux16_1tb.v",
    "mux32_1": "./mux32_1tb.v",
    "mux64_1": "./mux64_1tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./mux_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="ChatGPT",
        clk_signal=None,   # all of these mux modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,         # set to 0 if you want less diagnostic output
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        # Fallback to the last generated module file in that run directory.
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path


Optimizing: mux2_1 - 2-to-1 multiplexer
Using prev_modules:

========== PPA Iteration 0 / 10 ==========
Iteration 0: Accepted new best candidate
  Candidate metrics: transistor_est=12, num_cells=1, wire_bits=4
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 1 / 10 ==========
Iteration 1: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 2 / 10 ==========
Iteration 2: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=30, num_cells=3, wire_bits=6
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 3 / 10 ==========
Iteration 3: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7


---

# Part II — New hierarchical design: adder hierarchy

Design chosen: `half_adder → full_adder → adder4 → adder8` (Option A from the assignment PDF).

Each level instantiates the previous level: `full_adder` uses two `half_adder`s plus an OR, `adder4` chains four `full_adder`s with ripple carry, and `adder8` chains two `adder4`s. The top-level module (`adder8`) therefore instantiates every previously generated submodule either directly or transitively.

Baseline self-checking testbenches for each module are written below. They cover the obvious cases (zeros, max values, a few mid-range sums) but deliberately leave some bit-level toggles and logic combinations uncovered, so Part III's coverage-guided refinement has somewhere to improve.

In [37]:
#@title Write baseline self-checking testbenches for the adder hierarchy
import os

TB_HALF_ADDER = r"""`timescale 1ns/1ps
module half_adder_tb;
    reg a, b;
    wire sum, carry;
    integer errors;
    half_adder dut(.a(a), .b(b), .sum(sum), .carry(carry));
    initial begin
        $dumpfile("half_adder.vcd");
        $dumpvars(0, half_adder_tb);
        errors = 0;
        // Four input combinations
        a=0; b=0; #1;
        if (sum !== 0 || carry !== 0) begin $display("ERROR 00: sum=%b carry=%b", sum, carry); errors=errors+1; end
        else $display("Test 1 passed: 0+0");
        a=0; b=1; #1;
        if (sum !== 1 || carry !== 0) begin $display("ERROR 01: sum=%b carry=%b", sum, carry); errors=errors+1; end
        else $display("Test 2 passed: 0+1");
        a=1; b=0; #1;
        if (sum !== 1 || carry !== 0) begin $display("ERROR 10: sum=%b carry=%b", sum, carry); errors=errors+1; end
        else $display("Test 3 passed: 1+0");
        a=1; b=1; #1;
        if (sum !== 0 || carry !== 1) begin $display("ERROR 11: sum=%b carry=%b", sum, carry); errors=errors+1; end
        else $display("Test 4 passed: 1+1");

        if (errors == 0) $display("passed!");
        else $display("FAIL: %0d error(s)", errors);
        $finish;
    end
endmodule
"""

TB_FULL_ADDER = r"""`timescale 1ns/1ps
module full_adder_tb;
    reg a, b, cin;
    wire sum, cout;
    integer errors;
    full_adder dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
    initial begin
        $dumpfile("full_adder.vcd");
        $dumpvars(0, full_adder_tb);
        errors = 0;
        // All 8 input combinations
        a=0; b=0; cin=0; #1;
        if (sum !== 0 || cout !== 0) begin $display("ERROR 000"); errors=errors+1; end else $display("Test 1 passed: 0+0+0");
        a=0; b=0; cin=1; #1;
        if (sum !== 1 || cout !== 0) begin $display("ERROR 001"); errors=errors+1; end else $display("Test 2 passed: 0+0+1");
        a=0; b=1; cin=0; #1;
        if (sum !== 1 || cout !== 0) begin $display("ERROR 010"); errors=errors+1; end else $display("Test 3 passed: 0+1+0");
        a=0; b=1; cin=1; #1;
        if (sum !== 0 || cout !== 1) begin $display("ERROR 011"); errors=errors+1; end else $display("Test 4 passed: 0+1+1");
        a=1; b=0; cin=0; #1;
        if (sum !== 1 || cout !== 0) begin $display("ERROR 100"); errors=errors+1; end else $display("Test 5 passed: 1+0+0");
        a=1; b=0; cin=1; #1;
        if (sum !== 0 || cout !== 1) begin $display("ERROR 101"); errors=errors+1; end else $display("Test 6 passed: 1+0+1");
        a=1; b=1; cin=0; #1;
        if (sum !== 0 || cout !== 1) begin $display("ERROR 110"); errors=errors+1; end else $display("Test 7 passed: 1+1+0");
        a=1; b=1; cin=1; #1;
        if (sum !== 1 || cout !== 1) begin $display("ERROR 111"); errors=errors+1; end else $display("Test 8 passed: 1+1+1");

        if (errors == 0) $display("passed!");
        else $display("FAIL: %0d error(s)", errors);
        $finish;
    end
endmodule
"""

TB_ADDER4 = r"""`timescale 1ns/1ps
module adder4_tb;
    reg [3:0] a, b;
    reg cin;
    wire [3:0] sum;
    wire cout;
    integer errors;
    adder4 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
    initial begin
        $dumpfile("adder4.vcd");
        $dumpvars(0, adder4_tb);
        errors = 0;
        // Baseline cases - deliberately limited: a handful of common patterns
        a=4'h0; b=4'h0; cin=0; #1;
        if ({cout,sum} !== 5'h00) begin $display("ERR t1"); errors=errors+1; end else $display("Test 1 passed: 0+0");
        a=4'h3; b=4'h5; cin=0; #1;
        if ({cout,sum} !== 5'h08) begin $display("ERR t2"); errors=errors+1; end else $display("Test 2 passed: 3+5");
        a=4'hF; b=4'h1; cin=0; #1;
        if ({cout,sum} !== 5'h10) begin $display("ERR t3"); errors=errors+1; end else $display("Test 3 passed: F+1");
        a=4'hF; b=4'hF; cin=1; #1;
        if ({cout,sum} !== 5'h1F) begin $display("ERR t4"); errors=errors+1; end else $display("Test 4 passed: F+F+1");
        a=4'h7; b=4'h8; cin=0; #1;
        if ({cout,sum} !== 5'h0F) begin $display("ERR t5"); errors=errors+1; end else $display("Test 5 passed: 7+8");

        if (errors == 0) $display("passed!");
        else $display("FAIL: %0d error(s)", errors);
        $finish;
    end
endmodule
"""

TB_ADDER8 = r"""`timescale 1ns/1ps
module adder8_tb;
    reg [7:0] a, b;
    reg cin;
    wire [7:0] sum;
    wire cout;
    integer errors;
    adder8 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));
    initial begin
        $dumpfile("adder8.vcd");
        $dumpvars(0, adder8_tb);
        errors = 0;
        // Deliberately limited baseline - more exhaustive patterns added in Part III
        a=8'h00; b=8'h00; cin=0; #1;
        if ({cout,sum} !== 9'h000) begin $display("ERR t1"); errors=errors+1; end else $display("Test 1 passed: 0+0");
        a=8'h12; b=8'h34; cin=0; #1;
        if ({cout,sum} !== 9'h046) begin $display("ERR t2"); errors=errors+1; end else $display("Test 2 passed: 12+34");
        a=8'hFF; b=8'h01; cin=0; #1;
        if ({cout,sum} !== 9'h100) begin $display("ERR t3"); errors=errors+1; end else $display("Test 3 passed: FF+1 overflow");
        a=8'hFF; b=8'hFF; cin=1; #1;
        if ({cout,sum} !== 9'h1FF) begin $display("ERR t4"); errors=errors+1; end else $display("Test 4 passed: FF+FF+1");
        a=8'h55; b=8'hAA; cin=0; #1;
        if ({cout,sum} !== 9'h0FF) begin $display("ERR t5"); errors=errors+1; end else $display("Test 5 passed: 55+AA");

        if (errors == 0) $display("passed!");
        else $display("FAIL: %0d error(s)", errors);
        $finish;
    end
endmodule
"""

# Write the testbench files alongside the module names ROME expects
open("half_adder_tb.v", "w").write(TB_HALF_ADDER)
open("full_adder_tb.v", "w").write(TB_FULL_ADDER)
open("adder4_tb.v",     "w").write(TB_ADDER4)
open("adder8_tb.v",     "w").write(TB_ADDER8)

# Also save adder8 baseline (we'll iterate on this one during coverage refinement)
os.makedirs("coverage", exist_ok=True)
open("coverage/tb_baseline.v", "w").write(TB_ADDER8)

print("Baseline testbenches written:")
for f in ["half_adder_tb.v", "full_adder_tb.v", "adder4_tb.v", "adder8_tb.v"]:
    print("  " + f + " (" + str(len(open(f).read())) + " chars)")


Baseline testbenches written:
  half_adder_tb.v (1170 chars)
  full_adder_tb.v (1692 chars)
  adder4_tb.v (1275 chars)
  adder8_tb.v (1313 chars)


In [38]:
#@title Adder submodules definition (ROME submodule format)

### Each step: ["filename", "natural language description", "interface"]
adder_submodules = [
    ["half_adder",
     "A single-bit half adder. Compute the 1-bit sum of two 1-bit inputs a and b and the 1-bit carry out. sum = a XOR b. carry = a AND b.",
     "input a, input b, output sum, output carry"],

    ["full_adder",
     "A single-bit full adder. Compute the 1-bit sum of a, b, and cin, and the carry-out. You should instantiate TWO half_adder modules (already generated as half_adder.v) plus one OR gate for the carry-out: use the first half_adder on (a, b), feed its sum into the second half_adder with cin, use the second half_adder sum as the output sum, and OR the two half_adder carries to form cout.",
     "input a, input b, input cin, output sum, output cout"],

    ["adder4",
     "A 4-bit ripple-carry adder. Compute sum = a + b + cin as a 4-bit result with a 1-bit cout. You MUST instantiate FOUR full_adder modules (already generated as full_adder.v) chained via carry, i.e. ripple carry. Index bits with [i] starting from bit 0 (the LSB).",
     "input [3:0] a, input [3:0] b, input cin, output [3:0] sum, output cout"],

    ["adder8",
     "An 8-bit ripple-carry adder. Compute sum = a + b + cin as an 8-bit result with a 1-bit cout. You MUST instantiate TWO adder4 modules (already generated as adder4.v): the low adder4 handles bits [3:0] with cin, its cout feeds the high adder4 as cin, and the high adder4 handles bits [7:4]. The overall cout is the high adder4 cout.",
     "input [7:0] a, input [7:0] b, input cin, output [7:0] sum, output cout"],
]

print("Adder hierarchy ready, " + str(len(adder_submodules)) + " submodules")

Adder hierarchy ready, 4 submodules


In [39]:
#@title Run the ROME hierarchical loop on the adder design
# Same hier_gen() used for mux hierarchy in Part I
hier_gen(adder_submodules)


========== Iteration 0 / 10 ==========
VCD info: dumpfile half_adder.vcd opened for output.
Test 1 passed: 0+0
Test 2 passed: 0+1
Test 3 passed: 1+0
Test 4 passed: 1+1
passed!
./half_addertb.v:27: $finish called at 4000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  0.9776685237884521
Generation time:  0.9474201202392578
Error handling time:  0.030248165130615234
Module  1  done:  A single-bit half adder. Compute the 1-bit sum of two 1-bit inputs a and b and the 1-bit carry out. sum = a XOR b. carry = a AND b.

========== Iteration 0 / 10 ==========
VCD info: dumpfile full_adder.vcd opened for output.
Test 1 passed: 0+0+0
Test 2 passed: 0+0+1
Test 3 passed: 0+1+0
Test 4 passed: 0+1+1
Test 5 passed: 1+0+0
Test 6 passed: 1+0+1
Test 7 passed: 1+1+0
Test 8 passed: 1+1+1
passed!
./full_addertb.v:31: $finish called at 8000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Fi

(np.float64(8.512556791305542),
 np.float64(8.382953643798828),
 np.float64(0.12960195541381836))

---

# Part III — Integrate iVerilog + Covered coverage pipeline

Starting from the ROME workflow above, this section integrates the coverage-analysis pipeline from the course `coverage_exp` directory. The pipeline:

1. Compile DUT + current testbench with `iverilog -g2001`.
2. Run with `vvp` to produce a `.vcd` waveform.
3. Strip Icarus Verilog 13's `$comment` line that Covered 0.7.10 cannot parse.
4. Call `covered score ...` to build a `.cdd` database from the `.v` + `.vcd`.
5. Call `covered report -m ltc -d s|v` to produce line (l), toggle (t), and combinational-logic (c) reports.
6. Parse the summary to extract percentages; preserve the detailed report for the LLM.

The target DUT for coverage analysis is `adder8` (the top of the new hierarchy) — it has the richest combinational structure. We hold the RTL fixed throughout Part III and only modify the testbench. If a strengthened testbench ever exposes a new RTL failure, Step 4 of the assignment PDF kicks in and we return to the ROME loop with the TB frozen.

In [40]:
#@title Coverage pipeline helper functions
import os, re, json, subprocess, shutil, textwrap

COV_DIR = "coverage"
os.makedirs(COV_DIR, exist_ok=True)

def _run(cmd, cwd=".", timeout=60):
    """Run shell command, return (returncode, combined stdout+stderr)."""
    try:
        p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True,
                           text=True, timeout=timeout)
        return p.returncode, (p.stdout or "") + (p.stderr or "")
    except subprocess.TimeoutExpired:
        return -1, "TIMEOUT after " + str(timeout) + "s"

def collect_rtl_files():
    """Collect every .v file that adder8 transitively depends on, in dependency order."""
    files = []
    for name in ["half_adder", "full_adder", "adder4", "adder8"]:
        p = os.path.join(name, name + ".v")
        if os.path.exists(p):
            files.append(p)
    return files

def _strip_vcd_comment(src, dst):
    """Remove the $comment line iverilog 13 emits that Covered 0.7.10 cannot parse."""
    with open(src) as f:
        text = f.read()
    # Remove $comment ... $end lines
    text = re.sub(r'^\$comment [^\n]*\n', '', text, flags=re.M)
    # Normalize $timescale onto one line
    text = re.sub(r'\$timescale\s+1ps\s+\$end', '$timescale 1ps $end', text)
    with open(dst, "w") as f:
        f.write(text)

def run_coverage(tb_text, label):
    """
    Compile adder8 + full RTL + the given testbench, run vvp, run Covered.
    Returns a dict with percentages and the detailed report text.

    `label` is a short string used in filenames (e.g. "baseline", "iter_1").
    """
    workdir = os.path.join(COV_DIR, label)
    os.makedirs(workdir, exist_ok=True)

    # Copy RTL in
    rtl_files = collect_rtl_files()
    for rtl in rtl_files:
        shutil.copy(rtl, workdir)

    tb_path = os.path.join(workdir, "adder8_tb.v")
    open(tb_path, "w").write(tb_text)

    # Compile with iverilog
    rc, out_c = _run("iverilog -g2001 -Wall -s adder8_tb -o adder8.vvp "
                     "half_adder.v full_adder.v adder4.v adder8.v adder8_tb.v",
                     cwd=workdir, timeout=45)
    if rc != 0:
        return {"label": label, "compile_ok": False, "sim_ok": False,
                "line": None, "toggle_01": None, "toggle_10": None, "logic": None,
                "detail": "", "sim_output": out_c}

    # Simulate
    rc, out_s = _run("vvp adder8.vvp", cwd=workdir, timeout=60)
    sim_ok = ("passed!" in out_s) and ("FAIL" not in out_s)
    if not sim_ok:
        return {"label": label, "compile_ok": True, "sim_ok": False,
                "line": None, "toggle_01": None, "toggle_10": None, "logic": None,
                "detail": "", "sim_output": out_s}

    # Strip VCD comment so Covered 0.7.10 can read it
    _strip_vcd_comment(os.path.join(workdir, "adder8.vcd"),
                       os.path.join(workdir, "adder8_clean.vcd"))

    # Covered score
    rc, out_cov = _run("covered score -t adder8 -i adder8_tb.dut "
                       "-v half_adder.v -v full_adder.v -v adder4.v -v adder8.v "
                       "-vcd adder8_clean.vcd -o adder8.cdd",
                       cwd=workdir, timeout=60)
    if rc != 0:
        return {"label": label, "compile_ok": True, "sim_ok": True,
                "line": None, "toggle_01": None, "toggle_10": None, "logic": None,
                "detail": out_cov, "sim_output": out_s}

    # Reports
    _run("covered report -m ltc -d s -o coverage_summary.rpt adder8.cdd", cwd=workdir)
    _run("covered report -m ltc -d v -i -o coverage_detail.rpt adder8.cdd", cwd=workdir)

    summary_path = os.path.join(workdir, "coverage_summary.rpt")
    detail_path  = os.path.join(workdir, "coverage_detail.rpt")
    summary = open(summary_path).read() if os.path.exists(summary_path) else ""
    detail  = open(detail_path).read()  if os.path.exists(detail_path)  else ""

    percentages = parse_coverage_percentages(summary)
    return {
        "label": label, "compile_ok": True, "sim_ok": True,
        **percentages,
        "detail": detail, "summary": summary, "sim_output": out_s,
        "workdir": workdir,
    }

def parse_coverage_percentages(summary_text):
    """
    Parse Covered summary report for line / toggle / logic coverage percentages.

    Summary table in Covered looks like:
      Instance                   Hit/Miss/Total   Percent
      adder8_tb.dut              <nums>           <pct>
    for each metric section. We scan each section's totals rows.
    """
    def _pct(block):
        # Look for a 'Total' or 'adder8' summary line containing a percent
        # Fallback: last percent in the block.
        pcts = re.findall(r'(\d+(?:\.\d+)?)\s*%', block)
        if not pcts:
            return None
        return float(pcts[-1])

    def _hit_total(block):
        # Parse last 'hit total' pair in table rows
        rows = re.findall(r'(\d+)\s+(\d+)\s+\d+(?:\.\d+)?%', block)
        if not rows:
            return (None, None)
        hit, total = rows[-1]
        return (int(hit), int(total))

    # Covered sections start with headings like "LINE COVERAGE RESULTS",
    # "TOGGLE COVERAGE RESULTS", "COMBINATIONAL LOGIC COVERAGE RESULTS"
    out = {"line": None, "toggle_01": None, "toggle_10": None, "logic": None}

    line_section = re.search(r'LINE COVERAGE RESULTS(.*?)(?:TOGGLE COVERAGE|COMBINATIONAL|ASSERTION|$)',
                             summary_text, re.S)
    if line_section:
        out["line"] = _pct(line_section.group(1))

    toggle_section = re.search(r'TOGGLE COVERAGE RESULTS(.*?)(?:COMBINATIONAL|ASSERTION|FINITE|MEMORY|$)',
                               summary_text, re.S)
    if toggle_section:
        # Covered reports a single percent per instance that's the min of 0->1 and 1->0;
        # the detailed output usually has both. Try to extract both from toggle block.
        pcts = re.findall(r'(\d+(?:\.\d+)?)\s*%', toggle_section.group(1))
        if len(pcts) >= 2:
            out["toggle_01"] = float(pcts[-2])
            out["toggle_10"] = float(pcts[-1])
        elif len(pcts) == 1:
            out["toggle_01"] = out["toggle_10"] = float(pcts[-1])

    logic_section = re.search(r'COMBINATIONAL LOGIC COVERAGE RESULTS(.*?)(?:FINITE|MEMORY|ASSERTION|$)',
                              summary_text, re.S)
    if logic_section:
        out["logic"] = _pct(logic_section.group(1))

    return out

def extract_uncovered_targets(detail_text, max_chars=4000):
    """
    Extract the most actionable uncovered-target lines from Covered detail report:
      - uncovered RTL lines
      - signals missing a 0->1 or 1->0 toggle
      - uncovered combinational logic expressions
    Keep the output bounded so it fits in a prompt.
    """
    sections = {}
    for title, key in [("LINE COVERAGE", "line"),
                       ("TOGGLE COVERAGE", "toggle"),
                       ("COMBINATIONAL LOGIC COVERAGE", "logic")]:
        # Match from this title until the next ALL-CAPS COVERAGE header or EOF
        pat = re.compile(title + r'(.*?)(?:\n[A-Z]{3,}[^\n]*COVERAGE[^\n]*\n|$)', re.S)
        m = pat.search(detail_text)
        if m:
            sec = m.group(1)
            # Keep lines containing "Line No", "Signal", "Expression", "Missed", or percent < 100
            interesting = []
            for ln in sec.splitlines():
                if any(k in ln for k in ["Missed", "Missing", "missed", "Signal",
                                          "Expression", "Line", "1->0", "0->1",
                                          "LINE", "TOGGLE", "LOGIC"]):
                    interesting.append(ln)
            sections[key] = "\n".join(interesting)[:1500]
    joined = ""
    for k in ["line", "toggle", "logic"]:
        if sections.get(k):
            joined += "\n--- " + k.upper() + " uncovered details ---\n" + sections[k]
    return joined[:max_chars] if joined else "(no uncovered targets found in detail report)"

# Coverage history, in the exact format the assignment PDF asks for
coverage_history = []

def record(result, num_tests):
    coverage_history.append({
        "iteration": result["label"],
        "num_tests": num_tests,
        "line":      result["line"],
        "toggle_01": result["toggle_01"],
        "toggle_10": result["toggle_10"],
        "logic":     result["logic"],
        "sim_ok":    result["sim_ok"],
    })
    with open(os.path.join(COV_DIR, "coverage_history.json"), "w") as f:
        json.dump(coverage_history, f, indent=2)

print("Coverage helpers ready.")
print("  COV_DIR =", COV_DIR)


Coverage helpers ready.
  COV_DIR = coverage


In [41]:
#@title Baseline coverage measurement on adder8
baseline_tb = open("adder8_tb.v").read()
num_tests_baseline = baseline_tb.count("passed:")

print("Running baseline coverage with " + str(num_tests_baseline) + " tests ...")
baseline = run_coverage(baseline_tb, "baseline")

if not baseline["sim_ok"]:
    print("Baseline simulation did not pass:")
    print(baseline.get("sim_output", "")[-500:])
else:
    print("Baseline results:")
    for k in ["line", "toggle_01", "toggle_10", "logic"]:
        v = baseline.get(k)
        print("  " + k.ljust(10) + ": " + (str(v) + " %" if v is not None else "n/a"))

record(baseline, num_tests_baseline)
# Also keep the baseline TB in Part III's own folder for easy reference
import shutil
shutil.copy("adder8_tb.v", "coverage/tb_baseline.v")


Running baseline coverage with 5 tests ...
Baseline results:
  line      : 100.0 %
  toggle_01 : 100.0 %
  toggle_10 : 66.0 %
  logic     : 100.0 %


'coverage/tb_baseline.v'

In [42]:
#@title Coverage iteration 1 — LLM adds targeted tests for uncovered items
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

uncovered = extract_uncovered_targets(baseline["detail"])
print("--- UNCOVERED TARGETS (truncated) ---")
print(uncovered[:1200])
print("--- END ---")

current_tb = baseline_tb

prompt = (
    "You are refining a Verilog self-checking testbench to improve its code coverage.\n"
    "Keep ALL EXISTING TESTS EXACTLY as they are - do not delete or renumber them.\n"
    "Add NEW test cases targeted at the uncovered items reported by Covered below.\n"
    "The DUT is adder8(input [7:0] a, [7:0] b, input cin, output [7:0] sum, output cout).\n"
    "The testbench ends with: if (errors==0) $display(\"passed!\"); else $display(\"FAIL\"); $finish;\n"
    "Keep that structure intact - add new tests before the final if/else.\n"
    "Each new test MUST: set a,b,cin; wait #1; check {cout,sum} against the correct a+b+cin;\n"
    "on mismatch: increment errors and $display an ERROR message; on match: $display a test-pass line.\n"
    "Return ONLY the complete new testbench in one ```verilog block. No prose.\n\n"
    "=== CURRENT TESTBENCH ===\n```verilog\n" + current_tb + "\n```\n\n"
    "=== UNCOVERED TARGETS from Covered ===\n" + uncovered + "\n"
)

resp = client.chat.completions.create(
    model="gpt-4o-mini", temperature=0.3, max_tokens=3500,
    messages=[{"role":"user", "content": prompt}],
)
reply = resp.choices[0].message.content
import re
m = re.search(r"```(?:verilog|systemverilog|sv)?\s*\n(.*?)```", reply, re.S)
tb_iter1 = m.group(1).strip() if m else reply.strip()
open("coverage/tb_iter_1.v", "w").write(tb_iter1)

num_tests_iter1 = tb_iter1.count("passed:") + tb_iter1.count("passed!") - 1  # minus the final passed!
print("Iteration 1 TB written to coverage/tb_iter_1.v (" + str(len(tb_iter1)) + " chars)")


--- UNCOVERED TARGETS (truncated) ---

--- LINE uncovered details ---
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   TOGGLE COVERAGE RESULTS   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Signals not getting 100% toggle coverage
      Signal                    Toggle
      a                         0->1: 8'hff
      ......................... 1->0: 8'haa ...
      b                         0->1: 8'hff
      ......................... 1->0: 8'h75 ...
      sum                       0->1: 8'hff
      ......................... 1->0: 8'h46 ...
      sum_low                   0->1: 4'hf
      ......................... 1->0: 4'h6 ...
      sum_high                  0->1: 4'hf
      ......................... 1->0: 4'h4 ...
    Signals not getting 100% toggle coverage
      Signal                    Toggle
      a                         0->1: 4'hf
      ......................... 1->0: 4'ha ...
      b                         0->1: 4'hf
      ......................... 1->0: 4'h5 ...
      su

In [43]:
#@title Rerun coverage with iter_1 testbench
iter1 = run_coverage(tb_iter1, "iter_1")

if not iter1["sim_ok"]:
    print("Iteration 1 simulation did NOT pass with the refined testbench.")
    print("Per the assignment design rule: TB stays frozen, we would return to the ROME RTL-repair loop.")
    print("Sim output tail:")
    print(iter1.get("sim_output", "")[-800:])
else:
    print("Iteration 1 results:")
    for k in ["line", "toggle_01", "toggle_10", "logic"]:
        print("  " + k.ljust(10) + ": " + str(iter1.get(k)) + " %")

record(iter1, num_tests_iter1)


Iteration 1 results:
  line      : 100.0 %
  toggle_01 : 100.0 %
  toggle_10 : 100.0 %
  logic     : 100.0 %


In [47]:
#@title Coverage iteration 2 — one more refinement pass
uncovered2 = extract_uncovered_targets(iter1["detail"])
print("--- UNCOVERED TARGETS AFTER ITER 1 (truncated) ---")
print(uncovered2[:1200])

prompt2 = (
    "Here is the current testbench (iteration 1). Add MORE targeted test cases for the remaining "
    "uncovered items in the Covered report below. Keep all existing tests unchanged; only add new "
    "ones before the final if/else summary. Same rules as before: DUT is adder8, check cout+sum, "
    "increment errors on mismatch, keep passed!/FAIL summary intact. Return ONLY the complete "
    "testbench in one ```verilog block.\n\n"
    "=== CURRENT TESTBENCH ===\n```verilog\n" + tb_iter1 + "\n```\n\n"
    "=== UNCOVERED TARGETS ===\n" + uncovered2 + "\n"
)

resp = client.chat.completions.create(
    model="gpt-4o-mini", temperature=0.3, max_tokens=4000,
    messages=[{"role":"user", "content": prompt2}],
)
reply2 = resp.choices[0].message.content
import re
m = re.search(r"```(?:verilog|systemverilog|sv)?\s*\n(.*?)```", reply2, re.S)
tb_iter2 = m.group(1).strip() if m else reply2.strip()
open("coverage/tb_iter_2.v", "w").write(tb_iter2)

num_tests_iter2 = tb_iter2.count("passed:") + tb_iter2.count("passed!") - 1

iter2 = run_coverage(tb_iter2, "iter_2")
if not iter2["sim_ok"]:
    print("Iteration 2 simulation did NOT pass.")
    print(iter2.get("sim_output", "")[-800:])
else:
    print("Iteration 2 results:")
    for k in ["line", "toggle_01", "toggle_10", "logic"]:
        print("  " + k.ljust(10) + ": " + str(iter2.get(k)) + " %")

record(iter2, num_tests_iter2)

# Preserve the final (strongest) TB
import shutil
if iter2["sim_ok"]:
    shutil.copy("coverage/tb_iter_2.v", "coverage/tb_final.v")
elif iter1["sim_ok"]:
    shutil.copy("coverage/tb_iter_1.v", "coverage/tb_final.v")
else:
    shutil.copy("coverage/tb_baseline.v", "coverage/tb_final.v")


--- UNCOVERED TARGETS AFTER ITER 1 (truncated) ---

--- LINE uncovered details ---
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   TOGGLE COVERAGE RESULTS   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   COMBINATIONAL LOGIC COVERAGE RESULTS   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
--- TOGGLE uncovered details ---
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   COMBINATIONAL LOGIC COVERAGE RESULTS   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Iteration 2 results:
  line      : 100.0 %
  toggle_01 : 100.0 %
  toggle_10 : 100.0 %
  logic     : 100.0 %


---

# Part IV — Co-improvement summary and coverage history

The table below is written to `coverage/coverage_history.json` across all three runs.

In [48]:
#@title Print the coverage history + final iVerilog/vvp verification command
print("COVERAGE HISTORY")
print("-" * 70)
print(("Iteration".ljust(12) + "# tests".rjust(9) + "  Line".rjust(10) +
       "Tog0->1".rjust(10) + "Tog1->0".rjust(10) + "Logic".rjust(10)))
for row in coverage_history:
    def _f(v):
        if v is None: return "n/a".rjust(10)
        return (str(round(v,1)) + "%").rjust(10)
    print(row["iteration"].ljust(12)
          + str(row["num_tests"]).rjust(9)
          + _f(row["line"]) + _f(row["toggle_01"]) + _f(row["toggle_10"]) + _f(row["logic"]))

print()
print("Final iVerilog/vvp verification command (adder8 + strongest TB):")
print("  iverilog -g2001 -Wall -s adder8_tb -o adder8.vvp \\")
print("           half_adder.v full_adder.v adder4.v adder8.v coverage/tb_final.v")
print("  vvp adder8.vvp")

# Also package everything into a zip for easy download into HW_ROME_Coverage/
import zipfile, os
with zipfile.ZipFile("rome_submission_artifacts.zip", "w") as zf:
    # RTL + ROME iteration logs
    for mod in ["mux2_1","mux4_1","mux8_1","mux16_1","mux32_1","mux64_1",
                "half_adder","full_adder","adder4","adder8"]:
        if os.path.isdir(mod):
            for root, _, files in os.walk(mod):
                for f in files:
                    zf.write(os.path.join(root, f))
    # Testbench files
    for tb in ["half_adder_tb.v","full_adder_tb.v","adder4_tb.v","adder8_tb.v",
               "mux2_1tb.v","mux4_1tb.v","mux8_1tb.v","mux16_1tb.v","mux32_1tb.v","mux64_1tb.v"]:
        if os.path.exists(tb):
            zf.write(tb)
    # Coverage folder
    if os.path.isdir("coverage"):
        for root, _, files in os.walk("coverage"):
            for f in files:
                zf.write(os.path.join(root, f))

print()
print("Artifacts packaged: rome_submission_artifacts.zip")


COVERAGE HISTORY
----------------------------------------------------------------------
Iteration     # tests      Line   Tog0->1   Tog1->0     Logic
baseline            5    100.0%    100.0%     66.0%    100.0%
iter_1             10    100.0%    100.0%    100.0%    100.0%
iter_2             15       n/a       n/a       n/a       n/a
iter_2             15    100.0%    100.0%    100.0%    100.0%

Final iVerilog/vvp verification command (adder8 + strongest TB):
  iverilog -g2001 -Wall -s adder8_tb -o adder8.vvp \
           half_adder.v full_adder.v adder4.v adder8.v coverage/tb_final.v
  vvp adder8.vvp

Artifacts packaged: rome_submission_artifacts.zip


### Fix testbench filename convention and rerun the adder hierarchy

The baseline testbench cell above wrote files named `half_adder_tb.v`,
`full_adder_tb.v`, etc. (with an underscore before `tb`). ROME's
`hier_gen()` looks for `<module>tb.v` without that underscore (matching
the course convention used by `mux2_1tb.v` ... `mux64_1tb.v`). The cell
below renames each baseline testbench to the expected form, clears any
partial per-module folders left from the earlier compile-error attempts,
and re-runs `hier_gen(adder_submodules)` so Part II produces its final
`half_adder.v`, `full_adder.v`, `adder4.v`, and `adder8.v` RTL.

In [46]:
# Rename the baseline testbenches to the convention ROME expects (<module>tb.v)
import os, shutil
for src, dst in [("half_adder_tb.v", "half_addertb.v"),
                 ("full_adder_tb.v", "full_addertb.v"),
                 ("adder4_tb.v",     "adder4tb.v"),
                 ("adder8_tb.v",     "adder8tb.v")]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"copied {src} -> {dst}")

# Clean out the failed adder dirs so ROME regenerates from scratch
for d in ["half_adder", "full_adder", "adder4", "adder8"]:
    if os.path.isdir(d):
        shutil.rmtree(d)

# Rerun the adder hierarchy
hier_gen(adder_submodules)

copied half_adder_tb.v -> half_addertb.v
copied full_adder_tb.v -> full_addertb.v
copied adder4_tb.v -> adder4tb.v
copied adder8_tb.v -> adder8tb.v

========== Iteration 0 / 10 ==========
VCD info: dumpfile half_adder.vcd opened for output.
Test 1 passed: 0+0
Test 2 passed: 0+1
Test 3 passed: 1+0
Test 4 passed: 1+1
passed!
./half_addertb.v:27: $finish called at 4000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  1.0958905220031738
Generation time:  1.065760850906372
Error handling time:  0.030129432678222656
Module  1  done:  A single-bit half adder. Compute the 1-bit sum of two 1-bit inputs a and b and the 1-bit carry out. sum = a XOR b. carry = a AND b.

========== Iteration 0 / 10 ==========
VCD info: dumpfile full_adder.vcd opened for output.
Test 1 passed: 0+0+0
Test 2 passed: 0+0+1
Test 3 passed: 0+1+0
Test 4 passed: 0+1+1
Test 5 passed: 1+0+0
Test 6 passed: 1+0+1
Test 7 passed: 1+1+0
Test 8 p

(np.float64(7.485764503479004),
 np.float64(7.356741905212402),
 np.float64(0.12902116775512695))